# So sanh EfficientNet-B1 voi CBAM va Coordinate Attention tren Chest X-Ray

Notebook nay thuc hien phan loai anh X-ray nguc va so sanh cac bien the:
- EfficientNet-B1 baseline
- EfficientNet-B1 + CBAM (cuoi backbone / chen vao stage)
- EfficientNet-B1 + Coordinate Attention (cuoi backbone / chen vao stage)
- EfficientNet-B1 + CA + CBAM (ket hop)

Dataset: Chest X-Ray (Pneumonia, Covid-19, Tuberculosis, Normal)

## 1. Cai dat va import thu vien

In [ ]:
# Cai dat cac thu vien can thiet neu chua co
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

try:
    import torchinfo
except ImportError:
    pip_install("torchinfo")

try:
    from thop import profile as thop_profile
except ImportError:
    pip_install("thop")
    from thop import profile as thop_profile

print("Da cai xong cac thu vien can thiet.")

In [ ]:
import os
import time
import copy
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

from PIL import Image, UnidentifiedImageError

try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except ImportError:
    HAS_TORCHINFO = False
    print("Canh bao: khong co torchinfo, se in so tham so thu cong.")

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False
    print("Canh bao: khong co thop, FLOPs se la None.")

print("Import hoan tat.")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

## 2. Cau hinh thi nghiem

In [ ]:
# ---- Thiet bi ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Su dung thiet bi: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---- Seed co dinh de tai hien ket qua ----
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ---- Sieu tham so ----
IMG_SIZE       = 240        # EfficientNet-B1 dung anh 240x240
BATCH_SIZE     = 16
NUM_EPOCHS     = 20
LR             = 1e-4
WEIGHT_DECAY   = 1e-4
NUM_WORKERS    = 2
PATIENCE       = 5          # Early stopping

# ---- Thu muc luu ket qua ----
SAVE_DIR = "/kaggle/working/results"
GRADCAM_DIR = os.path.join(SAVE_DIR, "gradcam")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(GRADCAM_DIR, exist_ok=True)
print(f"Thu muc luu ket qua: {SAVE_DIR}")

## 3. Kiem tra va tim dataset

In [ ]:
def find_dataset_root(base="/kaggle/input"):
    """Tim thu muc chua ca 3 thu muc train, val, test."""
    for root, dirs, _ in os.walk(base):
        dirs_lower = [d.lower() for d in dirs]
        if "train" in dirs_lower and "val" in dirs_lower and "test" in dirs_lower:
            return root
    return None

DATA_ROOT = find_dataset_root()

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Khong tim thay thu muc chua train/val/test trong /kaggle/input. "
        "Hay kiem tra lai dataset da duoc them vao Kaggle chua."
    )

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

# -- Sua lai: tim chinh xac thu muc con co ten dung --
for split_name, split_attr in [("train", "TRAIN_DIR"), ("val", "VAL_DIR"), ("test", "TEST_DIR")]:
    path = os.path.join(DATA_ROOT, split_name)
    if not os.path.isdir(path):
        # Thu viet hoa
        for d in os.listdir(DATA_ROOT):
            if d.lower() == split_name:
                globals()[split_attr] = os.path.join(DATA_ROOT, d)

print(f"Data root : {DATA_ROOT}")
print(f"Train dir : {TRAIN_DIR}")
print(f"Val dir   : {VAL_DIR}")
print(f"Test dir  : {TEST_DIR}")

# Lay danh sach class tu thu muc train
CLASS_NAMES = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d))
])
NUM_CLASSES = len(CLASS_NAMES)
print(f"\nCac class ({NUM_CLASSES}): {CLASS_NAMES}")

In [ ]:
def count_images_per_class(split_dir, class_names):
    """Dem so anh trong moi class cua mot split."""
    counts = {}
    for cls in class_names:
        cls_path = os.path.join(split_dir, cls)
        if os.path.isdir(cls_path):
            imgs = [
                f for f in os.listdir(cls_path)
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
            ]
            counts[cls] = len(imgs)
        else:
            counts[cls] = 0
    return counts

train_counts = count_images_per_class(TRAIN_DIR, CLASS_NAMES)
val_counts   = count_images_per_class(VAL_DIR,   CLASS_NAMES)
test_counts  = count_images_per_class(TEST_DIR,  CLASS_NAMES)

print("So anh moi class:")
print(f"{'Class':<20} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 48)
for cls in CLASS_NAMES:
    print(f"{cls:<20} {train_counts[cls]:>8} {val_counts[cls]:>8} {test_counts[cls]:>8}")
print("-" * 48)
print(f"{'TONG'::<20} {sum(train_counts.values()):>8} {sum(val_counts.values()):>8} {sum(test_counts.values()):>8}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
splits = [("Train", train_counts), ("Val", val_counts), ("Test", test_counts)]

for ax, (split_name, counts) in zip(axes, splits):
    ax.bar(counts.keys(), counts.values(), color="steelblue", edgecolor="black")
    ax.set_title(f"So anh - {split_name}")
    ax.set_xlabel("Class")
    ax.set_ylabel("So luong anh")
    ax.tick_params(axis="x", rotation=30)
    for i, (cls, cnt) in enumerate(counts.items()):
        ax.text(i, cnt + 1, str(cnt), ha="center", va="bottom", fontsize=9)

plt.suptitle("Phan bo du lieu theo class", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "data_distribution.png"), dpi=100, bbox_inches="tight")
plt.show()
print("Da luu bieu do phan bo du lieu.")

In [ ]:
def check_corrupt_images(base_dir, class_names):
    """Kiem tra va bao cao anh bi loi."""
    corrupt = []
    for cls in class_names:
        cls_path = os.path.join(base_dir, cls)
        if not os.path.isdir(cls_path):
            continue
        for fname in os.listdir(cls_path):
            fpath = os.path.join(cls_path, fname)
            if not fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                continue
            try:
                with Image.open(fpath) as img:
                    img.verify()
            except Exception:
                corrupt.append(fpath)
    return corrupt

print("Dang kiem tra anh bi loi trong train...")
corrupt_train = check_corrupt_images(TRAIN_DIR, CLASS_NAMES)
print(f"Anh loi trong train: {len(corrupt_train)}")

print("Dang kiem tra anh bi loi trong val...")
corrupt_val = check_corrupt_images(VAL_DIR, CLASS_NAMES)
print(f"Anh loi trong val: {len(corrupt_val)}")

print("Dang kiem tra anh bi loi trong test...")
corrupt_test = check_corrupt_images(TEST_DIR, CLASS_NAMES)
print(f"Anh loi trong test: {len(corrupt_test)}")

if corrupt_train or corrupt_val or corrupt_test:
    print("\nDanh sach anh bi loi:")
    for p in (corrupt_train + corrupt_val + corrupt_test):
        print(f"  {p}")
else:
    print("\nKhong co anh bi loi.")

In [ ]:
def show_sample_images(split_dir, class_names, n_per_class=3, title=""):
    """Hien thi mot so anh mau tu moi class."""
    fig, axes = plt.subplots(
        len(class_names), n_per_class,
        figsize=(n_per_class * 3, len(class_names) * 3)
    )
    if len(class_names) == 1:
        axes = [axes]

    for row_idx, cls in enumerate(class_names):
        cls_path = os.path.join(split_dir, cls)
        imgs = [
            f for f in os.listdir(cls_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
        ]
        chosen = random.sample(imgs, min(n_per_class, len(imgs)))
        for col_idx in range(n_per_class):
            ax = axes[row_idx][col_idx] if n_per_class > 1 else axes[row_idx]
            if col_idx < len(chosen):
                img = Image.open(os.path.join(cls_path, chosen[col_idx])).convert("RGB")
                ax.imshow(img, cmap="gray")
                if col_idx == 0:
                    ax.set_ylabel(cls, fontsize=10, fontweight="bold")
            ax.axis("off")

    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"samples_{title.replace(' ', '_')}.png"),
                dpi=80, bbox_inches="tight")
    plt.show()

show_sample_images(TRAIN_DIR, CLASS_NAMES, n_per_class=3, title="Anh mau Train")

## 4. Tien xu ly du lieu va Data Augmentation

In [ ]:
# Augmentation cho tap train: lam phong phu du lieu de tranh overfit
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Khong augment val/test, chi resize va normalize
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Tao dataset voi ImageFolder (doc nhan tu ten thu muc)
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=val_test_transform)
test_dataset  = datasets.ImageFolder(TEST_DIR,  transform=val_test_transform)

# Tao DataLoader
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

# In thong tin dataset
print(f"So anh train   : {len(train_dataset)}")
print(f"So anh val     : {len(val_dataset)}")
print(f"So anh test    : {len(test_dataset)}")
print(f"Class to idx   : {train_dataset.class_to_idx}")
print(f"So batch train : {len(train_loader)}")

## 5. Xu ly mat can bang du lieu (class weights)

In [ ]:
# Dem so mau moi class trong tap train
labels_train = [label for _, label in train_dataset.samples]
class_sample_counts = np.bincount(labels_train, minlength=NUM_CLASSES)
print("So mau moi class trong train:")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  {cls}: {class_sample_counts[i]}")

# Tinh class weights: class it anh -> trong so cao hon
total_samples = sum(class_sample_counts)
class_weights = total_samples / (NUM_CLASSES * class_sample_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("\nClass weights:")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  {cls}: {class_weights[i]:.4f}")

# Ham loss co trong so de xu ly mat can bang
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
print("\nDa tao CrossEntropyLoss voi class weights.")

## 6. Cai dat CBAM (Convolutional Block Attention Module)

In [ ]:
class ChannelAttention(nn.Module):
    """
    Tinh attention theo chieu kenh (channel).
    Dung ca average pooling va max pooling, sau do chay qua MLP chung.
    """
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        # Dam bao so kenh sau khi giam khong qua nho
        mid = max(in_channels // reduction_ratio, 8)

        # MLP dung 2 lop Conv 1x1 (tuong duong Linear nhung giu chieu khong gian)
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, mid, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, in_channels, kernel_size=1, bias=False),
        )
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x):
        # Tinh dac trung tu hai nhanh pooling, cong lai roi qua sigmoid
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        scale   = self.sigmoid(avg_out + max_out)
        return x * scale  # Nhan vao input de chon loc kenh


class SpatialAttention(nn.Module):
    """
    Tinh attention theo chieu khong gian (spatial).
    Dung channel-wise avg va max, sau do conv 7x7.
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv    = nn.Conv2d(2, 1, kernel_size=kernel_size,
                                 padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Tinh trung binh va cuc dai theo chieu kenh
        avg_out = torch.mean(x, dim=1, keepdim=True)  # (B, 1, H, W)
        max_out, _ = torch.max(x, dim=1, keepdim=True)  # (B, 1, H, W)
        # Gom lai theo chieu kenh truoc khi conv
        concat = torch.cat([avg_out, max_out], dim=1)  # (B, 2, H, W)
        scale  = self.sigmoid(self.conv(concat))
        return x * scale  # Nhan vao input de chon loc vi tri


class CBAM(nn.Module):
    """
    CBAM: ap dung channel attention truoc, sau do spatial attention.
    Tham khao: Woo et al., ECCV 2018.
    """
    def __init__(self, in_channels, reduction_ratio=16, spatial_kernel=7):
        super().__init__()
        self.channel_att = ChannelAttention(in_channels, reduction_ratio)
        self.spatial_att = SpatialAttention(spatial_kernel)

    def forward(self, x):
        x = self.channel_att(x)  # Channel attention
        x = self.spatial_att(x)  # Spatial attention
        return x

print("Da dinh nghia CBAM (ChannelAttention + SpatialAttention).")

## 7. Cai dat Coordinate Attention

In [ ]:
class CoordinateAttention(nn.Module):
    """
    Coordinate Attention (CA): ma hoa thong tin vi tri theo 2 chieu H va W.
    Tham khao: Hou et al., CVPR 2021.

    Khac voi SE (chi dung global avg pool lam mat vi tri),
    CA pool theo tung chieu rieng biet nen giu duoc vi tri.
    """
    def __init__(self, in_channels, reduction_ratio=32):
        super().__init__()
        mid = max(in_channels // reduction_ratio, 8)

        # Lop conv giam chieu sau khi gop nhanh H va W
        self.conv_reduce = nn.Conv2d(in_channels, mid, kernel_size=1, bias=False)
        self.bn          = nn.BatchNorm2d(mid)
        self.act         = nn.Hardswish(inplace=True)  # Khuyen nghi trong bai bao

        # Hai lop conv phuc hoi chieu, mot cho H mot cho W
        self.conv_h = nn.Conv2d(mid, in_channels, kernel_size=1, bias=False)
        self.conv_w = nn.Conv2d(mid, in_channels, kernel_size=1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        B, C, H, W = x.shape

        # Pool theo chieu cao: giu H, gop W -> (B, C, H, 1)
        x_h = torch.mean(x, dim=3, keepdim=True)

        # Pool theo chieu rong: giu W, gop H -> (B, C, 1, W)
        # Sau do permute thanh (B, C, W, 1) de ghep voi x_h
        x_w = torch.mean(x, dim=2, keepdim=True).permute(0, 1, 3, 2)

        # Gop theo chieu spatial: (B, C, H+W, 1)
        y = torch.cat([x_h, x_w], dim=2)

        # Giam chieu kenh qua conv + BN + activation
        y = self.act(self.bn(self.conv_reduce(y)))

        # Tach lai thanh phan H va phan W
        x_h_feat, x_w_feat = torch.split(y, [H, W], dim=2)

        # Permute x_w_feat ve lai (B, mid, 1, W)
        x_w_feat = x_w_feat.permute(0, 1, 3, 2)

        # Tao attention map cho H va W qua sigmoid
        a_h = self.sigmoid(self.conv_h(x_h_feat))   # (B, C, H, 1)
        a_w = self.sigmoid(self.conv_w(x_w_feat))   # (B, C, 1, W)

        # Nhan vao input: quan tam theo ca chieu H lan W
        return x * a_h * a_w

print("Da dinh nghia Coordinate Attention.")

## 8. Xay dung model EfficientNet-B1 voi Attention

In [ ]:
def infer_feature_channels(backbone_features, img_size=240, device=DEVICE):
    """
    Dua dummy input qua tung features[i] de lay so channels va spatial size.
    Tra ve dict: index -> (channels, H, W).
    """
    info = {}
    x = torch.zeros(1, 3, img_size, img_size).to(device)
    backbone_features = backbone_features.to(device)
    with torch.no_grad():
        for i, layer in enumerate(backbone_features):
            x = layer(x)
            info[i] = (x.shape[1], x.shape[2], x.shape[3])
    return info

# Tao backbone tam thoi de do channel
_temp_backbone = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.IMAGENET1K_V1)
channel_info = infer_feature_channels(_temp_backbone.features)
del _temp_backbone

print("Thong tin channel va spatial size sau tung features[i] cua EfficientNet-B1:")
print(f"{'Index':<8} {'Channels':>10} {'H':>6} {'W':>6}")
print("-" * 34)
for i, (c, h, w) in channel_info.items():
    print(f"  [{i}]    {c:>10}   {h:>4}   {w:>4}")

CH_STAGE6 = channel_info[6][0]
CH_STAGE7 = channel_info[7][0]
CH_LAST   = channel_info[8][0]
print(f"\nSo channels dung cho attention:")
print(f"  Sau features[6]: {CH_STAGE6}")
print(f"  Sau features[7]: {CH_STAGE7}")
print(f"  Sau features[8] (last): {CH_LAST}")

In [ ]:
class FlexibleEfficientNetB1(nn.Module):
    """
    EfficientNet-B1 voi cac che do them attention:
      - baseline      : khong co attention
      - cbam_last     : CBAM sau toan bo features
      - ca_last       : Coordinate Attention sau toan bo features
      - ca_cbam_last  : CA + CBAM sau toan bo features
      - cbam_stage    : CBAM chen vao sau features[6] va features[7]
      - ca_stage      : CA chen vao sau features[6] va features[7]
      - ca_cbam_stage : CA sau features[6], CBAM sau features[7]
    """

    VALID_MODES = [
        "baseline", "cbam_last", "ca_last", "ca_cbam_last",
        "cbam_stage", "ca_stage", "ca_cbam_stage"
    ]

    def __init__(self, num_classes=NUM_CLASSES, mode="baseline"):
        super().__init__()
        assert mode in self.VALID_MODES, f"mode phai la mot trong {self.VALID_MODES}"
        self.mode = mode

        # Tai pretrained EfficientNet-B1
        backbone = models.efficientnet_b1(
            weights=models.EfficientNet_B1_Weights.IMAGENET1K_V1
        )

        # Luu tung block features de co the chen attention vao giua
        self.features = backbone.features
        self.avgpool  = backbone.avgpool

        # Thay classifier: dropout + linear
        in_features = backbone.classifier[1].in_features
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(in_features, num_classes)
        )

        # Tao cac module attention tuong ung
        if mode == "cbam_last":
            self.att_last = CBAM(CH_LAST)

        elif mode == "ca_last":
            self.att_last = CoordinateAttention(CH_LAST)

        elif mode == "ca_cbam_last":
            self.att_ca   = CoordinateAttention(CH_LAST)
            self.att_cbam = CBAM(CH_LAST)

        elif mode == "cbam_stage":
            self.att_s6 = CBAM(CH_STAGE6)
            self.att_s7 = CBAM(CH_STAGE7)

        elif mode == "ca_stage":
            self.att_s6 = CoordinateAttention(CH_STAGE6)
            self.att_s7 = CoordinateAttention(CH_STAGE7)

        elif mode == "ca_cbam_stage":
            self.att_s6 = CoordinateAttention(CH_STAGE6)
            self.att_s7 = CBAM(CH_STAGE7)

    def forward(self, x):
        if self.mode == "baseline":
            # Chay toan bo features, khong them attention
            x = self.features(x)

        elif self.mode == "cbam_last":
            x = self.features(x)
            x = self.att_last(x)

        elif self.mode == "ca_last":
            x = self.features(x)
            x = self.att_last(x)

        elif self.mode == "ca_cbam_last":
            x = self.features(x)
            x = self.att_ca(x)
            x = self.att_cbam(x)

        elif self.mode in ("cbam_stage", "ca_stage", "ca_cbam_stage"):
            # Chay tung buoc, chen attention sau stage 6 va stage 7
            for i in range(7):          # features[0] den features[6]
                x = self.features[i](x)
            x = self.att_s6(x)          # Attention sau features[6]
            x = self.features[7](x)     # features[7]
            x = self.att_s7(x)          # Attention sau features[7]
            x = self.features[8](x)     # features[8] = final conv

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


print("Da dinh nghia class FlexibleEfficientNetB1.")

# Kiem tra thu tung mode
print("\nKiem tra build model:")
for m in FlexibleEfficientNetB1.VALID_MODES:
    mdl = FlexibleEfficientNetB1(num_classes=NUM_CLASSES, mode=m).to(DEVICE)
    with torch.no_grad():
        out = mdl(torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE))
    print(f"  mode={m:<20} output shape: {out.shape}")
    del mdl
torch.cuda.empty_cache()

## 9. In thong tin model

In [ ]:
def count_params(model):
    """Dem so tham so cua model."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def get_flops(model, img_size=IMG_SIZE, device=DEVICE):
    """
    Tinh FLOPs bang thop tren ban COPY cua model.
    Quan trong: thop.profile() inject total_ops/total_params vao state_dict
    cua model goc, lam hong checkpoint neu goi truoc khi save.
    Giai phap: deep copy model, tinh FLOPs tren ban copy, xoa copy.
    """
    if not HAS_THOP:
        return None
    try:
        import copy
        # Tao ban sao rieng biet, khong anh huong model goc
        model_copy = copy.deepcopy(model)
        model_copy.eval()
        dummy = torch.zeros(1, 3, img_size, img_size).to(device)
        macs, _ = thop_profile(model_copy, inputs=(dummy,), verbose=False)
        flops   = macs * 2  # 1 MAC = 2 FLOPs
        del model_copy
        torch.cuda.empty_cache()
        return flops
    except Exception as e:
        print(f"  Canh bao: khong tinh duoc FLOPs ({e})")
        return None

def print_model_info(model, model_name, img_size=IMG_SIZE, device=DEVICE):
    """In day du thong tin cua mot model."""
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")

    total, trainable = count_params(model)
    non_trainable = total - trainable
    print(f"Tong so tham so        : {total:,}")
    print(f"Tham so co the train   : {trainable:,}")
    print(f"Tham so khong train    : {non_trainable:,}")

    # Tinh FLOPs tren ban copy -> model goc khong bi nhiem total_ops/total_params
    flops = get_flops(model, img_size, device)
    if flops is not None:
        print(f"FLOPs (approx.)        : {flops/1e9:.2f} GFLOPs")
    else:
        print("FLOPs                  : Khong tinh duoc")

    if HAS_TORCHINFO:
        try:
            print("\nTorchInfo Summary:")
            torchinfo_summary(
                model,
                input_size=(1, 3, img_size, img_size),
                device=device,
                verbose=0,
                col_names=["input_size", "output_size", "num_params"],
            )
        except Exception as e:
            print(f"  Canh bao torchinfo: {e}")

    return total, trainable, flops

# Kiem tra: model goc phai sach sau khi tinh FLOPs
mdl_demo = FlexibleEfficientNetB1(num_classes=NUM_CLASSES, mode="baseline").to(DEVICE)
_ = print_model_info(mdl_demo, "EfficientNet-B1 Baseline")

# Xac nhan state_dict khong co key la cua thop
keys_with_ops = [k for k in mdl_demo.state_dict().keys() if "total_ops" in k or "total_params" in k]
if keys_with_ops:
    print(f"\nCANH BAO: state_dict bi nhiem {len(keys_with_ops)} keys cua thop!")
else:
    print("\nKiem tra sach: state_dict khong co key nao cua thop.")

del mdl_demo
torch.cuda.empty_cache()


## 10. Ham train va validate

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    """Chay mot epoch train, tra ve (loss trung binh, accuracy)."""
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds         = outputs.argmax(dim=1)
        correct      += (preds == labels).sum().item()
        total        += images.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion):
    """Danh gia model tren mot loader, tra ve dict cac metrics."""
    model.eval()
    running_loss = 0.0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds         = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss  = running_loss / len(loader.dataset)
    acc       = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall    = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1        = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return {
        "loss": avg_loss, "acc": acc,
        "precision": precision, "recall": recall, "f1": f1,
        "preds": all_preds, "labels": all_labels
    }


def train_model(model, model_name, num_epochs=NUM_EPOCHS, patience=PATIENCE):
    """
    Vong lap train chinh:
    - Dung AdamW + CosineAnnealingLR
    - Early stopping theo val_f1
    - Luu best model theo val_f1 (luu ngay sau epoch 1 de dam bao luon co file)
    Tra ve: history dict, best_val_f1, best_val_acc, thoi gian train, best_path
    """
    optimizer = optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6
    )

    best_val_f1   = -1.0   # Bat dau tu -1 de dam bao epoch 1 luon duoc luu
    best_val_acc  = 0.0
    no_improve    = 0
    best_path     = os.path.join(SAVE_DIR, f"best_{model_name}.pth")

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss":   [], "val_acc":   [],
        "val_f1":     [], "val_precision": [], "val_recall": []
    }

    print(f"\nBat dau train: {model_name}")
    print(f"{'Epoch':<8} {'Train Loss':<12} {'Train Acc':<12} "
          f"{'Val Loss':<12} {'Val Acc':<12} {'Val F1':<10} {'LR':<12}")
    print("-" * 78)

    start_time = time.time()

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_metrics = evaluate(model, val_loader, criterion)

        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["acc"])
        history["val_f1"].append(val_metrics["f1"])
        history["val_precision"].append(val_metrics["precision"])
        history["val_recall"].append(val_metrics["recall"])

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"{epoch:<8} {train_loss:<12.4f} {train_acc:<12.4f} "
              f"{val_metrics['loss']:<12.4f} {val_metrics['acc']:<12.4f} "
              f"{val_metrics['f1']:<10.4f} {current_lr:<12.6f}")

        # Luu model tot nhat theo val_f1
        if val_metrics["f1"] > best_val_f1:
            best_val_f1  = val_metrics["f1"]
            best_val_acc = val_metrics["acc"]
            no_improve   = 0
            # Luu ca mode de load lai chinh xac
            torch.save(
                {
                    "state_dict": model.state_dict(),
                    "mode":       model.mode,
                    "epoch":      epoch,
                    "val_f1":     best_val_f1,
                },
                best_path
            )
            print(f"  -> Luu checkpoint moi: val_f1={best_val_f1:.4f}")
        else:
            no_improve += 1

        # Dung som neu khong cai thien
        if no_improve >= patience:
            print(f"  Early stopping tai epoch {epoch} (patience={patience})")
            break

    # Phuong an du phong: neu best_path khong ton tai (khong bao gio luu duoc)
    # thi luu model hien tai de tranh loi khi load
    if not os.path.exists(best_path):
        print(f"  Canh bao: chua co checkpoint, luu model hien tai vao {best_path}")
        torch.save(
            {
                "state_dict": model.state_dict(),
                "mode":       model.mode,
                "epoch":      num_epochs,
                "val_f1":     best_val_f1,
            },
            best_path
        )
        best_val_f1  = max(best_val_f1, 0.0)
        best_val_acc = max(best_val_acc, 0.0)

    train_time = time.time() - start_time
    print(f"\nKet qua train: best_val_f1={best_val_f1:.4f}, "
          f"best_val_acc={best_val_acc:.4f}")
    print(f"Thoi gian train: {train_time:.1f} giay")
    print(f"Da luu best model vao: {best_path}")

    return history, best_val_f1, best_val_acc, train_time, best_path

print("Da dinh nghia cac ham train/evaluate.")


## 11. Chay toan bo thi nghiem

In [ ]:
# Danh sach cac model can thi nghiem
# baseline chi train mot lan, dung lam nen cho ca Case A va Case B
EXPERIMENT_LIST = [
    ("EfficientNet_B1_Baseline",    "baseline"),
    ("EfficientNet_B1_CBAM_Last",   "cbam_last"),
    ("EfficientNet_B1_CA_Last",     "ca_last"),
    ("EfficientNet_B1_CA_CBAM_Last","ca_cbam_last"),
    ("EfficientNet_B1_CBAM_Stage",  "cbam_stage"),
    ("EfficientNet_B1_CA_Stage",    "ca_stage"),
    ("EfficientNet_B1_CA_CBAM_Stage","ca_cbam_stage"),
]

# Luu ket qua tong hop
results_list    = []
histories_all   = {}

for model_name, mode in EXPERIMENT_LIST:
    set_seed()
    print(f"\n{'#'*60}")
    print(f"# Thi nghiem: {model_name}")
    print(f"{'#'*60}")

    # Tao model moi
    model = FlexibleEfficientNetB1(num_classes=NUM_CLASSES, mode=mode).to(DEVICE)

    # In thong tin model
    total_params, trainable_params, flops = print_model_info(model, model_name)

    # Train
    history, best_val_f1, best_val_acc, train_time, best_path = train_model(
        model, model_name
    )
    histories_all[model_name] = history

    # Ghi lai ket qua ban dau (test se duoc cap nhat o phan sau)
    results_list.append({
        "model_name":         model_name,
        "mode":               mode,
        "attention_position": "last" if "last" in mode else ("stage" if "stage" in mode else "none"),
        "params_total":       total_params,
        "params_trainable":   trainable_params,
        "flops":              flops,
        "best_val_acc":       best_val_acc,
        "best_val_f1":        best_val_f1,
        "train_time_seconds": train_time,
        "checkpoint_path":    best_path,
        # Cac truong test se cap nhat sau
        "test_acc":           None,
        "test_precision":     None,
        "test_recall":        None,
        "test_f1":            None,
        "inference_time_ms":  None,
    })

    # Giai phong bo nho GPU
    del model
    torch.cuda.empty_cache()
    print(f"Da xong: {model_name}")

print("\nHoan thanh toan bo thi nghiem!")

## 12. Danh gia tren tap test va ve confusion matrix

In [ ]:
def evaluate_on_test(model_name, mode, checkpoint_path):
    """
    Load best checkpoint va danh gia tren test set.
    Loc sach cac key rac cua thop (total_ops, total_params) truoc khi load.
    """
    if not os.path.exists(checkpoint_path):
        print(f"LOI: Khong tim thay checkpoint: {checkpoint_path}")
        return {"test_acc": None, "test_precision": None,
                "test_recall": None, "test_f1": None,
                "inference_time_ms": None}

    model = FlexibleEfficientNetB1(num_classes=NUM_CLASSES, mode=mode).to(DEVICE)

    try:
        raw = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

        # Lay state_dict du la dict boc ngoai hay thuan
        if isinstance(raw, dict) and "state_dict" in raw:
            state_dict = raw["state_dict"]
        else:
            state_dict = raw

        # Loc bo tat ca key do thop inject vao (total_ops, total_params)
        # Day la nguyen nhan goc cua RuntimeError "Unexpected key(s)"
        clean_sd = {
            k: v for k, v in state_dict.items()
            if "total_ops" not in k and "total_params" not in k
        }
        n_removed = len(state_dict) - len(clean_sd)
        if n_removed > 0:
            print(f"  Da loc bo {n_removed} key rac cua thop khoi state_dict.")

        missing, unexpected = model.load_state_dict(clean_sd, strict=True)
        if missing:
            print(f"  Canh bao - Keys bi thieu: {missing[:3]}")
        if unexpected:
            print(f"  Canh bao - Keys thua con lai: {unexpected[:3]}")

    except Exception as e:
        print(f"  LOI khi load checkpoint: {e}")
        del model
        torch.cuda.empty_cache()
        return {"test_acc": None, "test_precision": None,
                "test_recall": None, "test_f1": None,
                "inference_time_ms": None}

    model.eval()
    all_preds  = []
    all_labels = []
    total_time = 0.0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            t0     = time.time()
            out    = model(images)
            total_time += time.time() - t0
            preds  = out.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc       = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall    = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1        = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    inf_ms    = (total_time / max(len(test_dataset), 1)) * 1000

    print(f"\n{'-'*50}")
    print(f"Ket qua test: {model_name}")
    print(f"  Accuracy        : {acc:.4f}")
    print(f"  Precision macro : {precision:.4f}")
    print(f"  Recall macro    : {recall:.4f}")
    print(f"  F1 macro        : {f1:.4f}")
    print(f"  Inference time  : {inf_ms:.3f} ms/anh")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds,
                                target_names=CLASS_NAMES, zero_division=0))

    # Ve confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"Confusion Matrix - {model_name}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    cm_path = os.path.join(SAVE_DIR, f"cm_{model_name}.png")
    plt.savefig(cm_path, dpi=100, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"  Da luu confusion matrix: {cm_path}")

    del model
    torch.cuda.empty_cache()

    return {"test_acc": acc, "test_precision": precision,
            "test_recall": recall, "test_f1": f1,
            "inference_time_ms": inf_ms}


# Danh gia tung model va cap nhat results_list
for entry in results_list:
    print(f"\nDang danh gia: {entry['model_name']}")
    test_metrics = evaluate_on_test(
        entry["model_name"],
        entry["mode"],
        entry["checkpoint_path"]
    )
    entry.update(test_metrics)

print("\nHoan thanh danh gia tren test set.")


## 13. Luu ket qua ra CSV

In [ ]:
# Tao DataFrame tong hop
results_df = pd.DataFrame(results_list)

# Luu summary
summary_path = os.path.join(SAVE_DIR, "summary_results.csv")
results_df.to_csv(summary_path, index=False)
print(f"Da luu summary: {summary_path}")
print(results_df[["model_name", "test_acc", "test_f1",
                   "params_total", "flops",
                   "train_time_seconds", "inference_time_ms"]].to_string())

# Luu history train cua tung model
history_records = []
for model_name, hist in histories_all.items():
    for epoch_idx in range(len(hist["train_loss"])):
        history_records.append({
            "model_name":  model_name,
            "epoch":       epoch_idx + 1,
            "train_loss":  hist["train_loss"][epoch_idx],
            "train_acc":   hist["train_acc"][epoch_idx],
            "val_loss":    hist["val_loss"][epoch_idx],
            "val_acc":     hist["val_acc"][epoch_idx],
            "val_f1":      hist["val_f1"][epoch_idx],
        })

history_df = pd.DataFrame(history_records)
history_path = os.path.join(SAVE_DIR, "training_history.csv")
history_df.to_csv(history_path, index=False)
print(f"Da luu history: {history_path}")

## 14. Ve bieu do ket qua

In [ ]:
def plot_training_curves(histories, save_dir=SAVE_DIR):
    """Ve loss va accuracy theo epoch cho tung model."""
    for model_name, hist in histories.items():
        epochs = range(1, len(hist["train_loss"]) + 1)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Loss
        axes[0].plot(epochs, hist["train_loss"], label="Train Loss", color="royalblue")
        axes[0].plot(epochs, hist["val_loss"],   label="Val Loss",   color="tomato")
        axes[0].set_title(f"Loss - {model_name}")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].legend()
        axes[0].grid(True, linestyle="--", alpha=0.6)

        # Accuracy
        axes[1].plot(epochs, hist["train_acc"], label="Train Acc", color="royalblue")
        axes[1].plot(epochs, hist["val_acc"],   label="Val Acc",   color="tomato")
        axes[1].set_title(f"Accuracy - {model_name}")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Accuracy")
        axes[1].legend()
        axes[1].grid(True, linestyle="--", alpha=0.6)

        plt.suptitle(model_name, fontsize=11, fontweight="bold")
        plt.tight_layout()
        save_path = os.path.join(save_dir, f"curve_{model_name}.png")
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
        plt.show()
        plt.close()

plot_training_curves(histories_all)
print("Da luu bieu do training curves.")

In [ ]:
def plot_comparison_bars(df, save_dir=SAVE_DIR):
    """Ve bar chart so sanh cac model theo nhieu tieu chi."""
    model_names = df["model_name"].tolist()
    short_names = [n.replace("EfficientNet_B1_", "").replace("_", "\n") for n in model_names]
    x = np.arange(len(model_names))
    bar_w = 0.6

    metrics = [
        ("test_acc",           "Test Accuracy",     "steelblue"),
        ("test_f1",            "Test F1 Macro",     "darkorange"),
        ("params_trainable",   "Trainable Params",  "mediumseagreen"),
        ("train_time_seconds", "Train Time (s)",    "mediumpurple"),
        ("inference_time_ms",  "Inference (ms/img)","crimson"),
    ]

    for col, ylabel, color in metrics:
        if col not in df.columns:
            continue
        vals = df[col].fillna(0).tolist()
        fig, ax = plt.subplots(figsize=(max(10, len(model_names) * 1.5), 5))
        bars = ax.bar(x, vals, width=bar_w, color=color, edgecolor="black")
        ax.set_xticks(x)
        ax.set_xticklabels(short_names, fontsize=9)
        ax.set_ylabel(ylabel)
        ax.set_title(f"So sanh cac model - {ylabel}")
        ax.grid(axis="y", linestyle="--", alpha=0.6)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(vals) * 0.01,
                    f"{v:.4f}" if v < 100 else f"{int(v):,}",
                    ha="center", va="bottom", fontsize=8)
        plt.tight_layout()
        fname = col.replace(" ", "_")
        plt.savefig(os.path.join(save_dir, f"compare_{fname}.png"),
                    dpi=100, bbox_inches="tight")
        plt.show()
        plt.close()

plot_comparison_bars(results_df)

# Ve FLOPs rieng vi co the la None
if "flops" in results_df.columns:
    flops_df = results_df[results_df["flops"].notna()].copy()
    if not flops_df.empty:
        flops_df["flops_G"] = flops_df["flops"] / 1e9
        short_names = [n.replace("EfficientNet_B1_", "").replace("_", "\n")
                       for n in flops_df["model_name"]]
        fig, ax = plt.subplots(figsize=(max(10, len(flops_df) * 1.5), 5))
        bars = ax.bar(range(len(flops_df)), flops_df["flops_G"],
                      color="teal", edgecolor="black")
        ax.set_xticks(range(len(flops_df)))
        ax.set_xticklabels(short_names, fontsize=9)
        ax.set_ylabel("GFLOPs")
        ax.set_title("So sanh FLOPs cac model")
        ax.grid(axis="y", linestyle="--", alpha=0.6)
        for bar, v in zip(bars, flops_df["flops_G"]):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() * 1.01,
                    f"{v:.2f}G", ha="center", va="bottom", fontsize=8)
        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_DIR, "compare_flops.png"),
                    dpi=100, bbox_inches="tight")
        plt.show()
        plt.close()

print("Da luu tat ca bieu do so sanh.")

## 15. Grad-CAM - Truc quan hoa vung anh mo hinh tap trung

In [ ]:
# Grad-CAM giup hieu model dang nhin vao vung nao cua anh X-ray
# khi du doan benh. Day la cong cu quan trong de giai thich model.

class GradCAM:
    """
    Grad-CAM: dung gradient cua lop conv cuoi de tao heatmap chi ra
    vung nao tren anh anh huong nhieu nhat toi quyet dinh cua model.
    """
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, class_idx=None):
        """Tao heatmap Grad-CAM cho mot anh."""
        self.model.eval()
        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        # Tinh trong so alpha: trung binh gradient theo khong gian
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)

        # Tinh ban do kech hoat co trong so
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)  # Chi giu gia tri duong

        # Normalize ve [0, 1]
        cam_min = cam.min()
        cam_max = cam.max()
        if cam_max - cam_min > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)

        return cam.squeeze().cpu().numpy(), class_idx


def get_gradcam_target_layer(model):
    """Lay lop conv cuoi cua EfficientNet features de lam target cho Grad-CAM."""
    # features[8] la conv cuoi truoc avgpool trong EfficientNet-B1
    return model.features[8]


def show_gradcam(model, model_name, mode, checkpoint_path,
                 n_images=4, save_dir=GRADCAM_DIR):
    """Hien thi va luu Grad-CAM cho n_images tu test set."""
    raw = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    state_dict = raw["state_dict"] if isinstance(raw, dict) and "state_dict" in raw else raw
    # Loc key rac cua thop
    state_dict = {k: v for k, v in state_dict.items()
                  if "total_ops" not in k and "total_params" not in k}
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    target_layer = get_gradcam_target_layer(model)
    gradcam      = GradCAM(model, target_layer)

    # Lay mot batch tu test loader
    images, labels = next(iter(test_loader))
    images = images[:n_images]
    labels = labels[:n_images]

    fig, axes = plt.subplots(n_images, 3, figsize=(12, n_images * 3))
    if n_images == 1:
        axes = [axes]

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    for idx in range(n_images):
        inp        = images[idx:idx+1].to(DEVICE)
        cam, pred  = gradcam.generate(inp)

        # Lay anh goc (de normalize)
        img_np = images[idx].permute(1, 2, 0).numpy()
        img_np = np.clip(img_np * std + mean, 0, 1)

        # Resize CAM ve kich thuoc anh
        from PIL import Image as PILImage
        cam_pil  = PILImage.fromarray((cam * 255).astype(np.uint8))
        cam_resized = np.array(cam_pil.resize(
            (IMG_SIZE, IMG_SIZE), PILImage.BILINEAR
        )) / 255.0

        # Overlay: ket hop anh goc va heatmap
        heatmap = plt.cm.jet(cam_resized)[:, :, :3]
        overlay = img_np * 0.5 + heatmap * 0.5

        true_cls = CLASS_NAMES[labels[idx].item()]
        pred_cls = CLASS_NAMES[pred]
        color    = "green" if pred == labels[idx].item() else "red"

        axes[idx][0].imshow(img_np)
        axes[idx][0].set_title(f"Anh goc\nTrue: {true_cls}", fontsize=9)
        axes[idx][0].axis("off")

        axes[idx][1].imshow(cam_resized, cmap="jet")
        axes[idx][1].set_title("Grad-CAM Heatmap", fontsize=9)
        axes[idx][1].axis("off")

        axes[idx][2].imshow(overlay)
        axes[idx][2].set_title(f"Overlay\nPred: {pred_cls}", fontsize=9,
                               color=color)
        axes[idx][2].axis("off")

    plt.suptitle(f"Grad-CAM - {model_name}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    save_path = os.path.join(save_dir, f"gradcam_{model_name}.png")
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Da luu Grad-CAM: {save_path}")


# Chay Grad-CAM cho 3 model theo yeu cau
gradcam_targets = [
    ("EfficientNet_B1_Baseline",  "baseline"),
    ("EfficientNet_B1_CA_Last",   "ca_last"),
    ("EfficientNet_B1_CA_Stage",  "ca_stage"),
]

for model_name, mode in gradcam_targets:
    # Tim checkpoint tuong ung
    entry = next((e for e in results_list if e["model_name"] == model_name), None)
    if entry is None:
        print(f"Khong tim thay entry cho {model_name}, bo qua.")
        continue

    model = FlexibleEfficientNetB1(num_classes=NUM_CLASSES, mode=mode).to(DEVICE)
    show_gradcam(model, model_name, mode,
                 entry["checkpoint_path"], n_images=4)
    del model
    torch.cuda.empty_cache()

print("\nHoan thanh Grad-CAM.")

## 16. Ket luan tu dong dua tren ket qua thuc te

In [ ]:
print("\n" + "="*60)
print("KET LUAN")
print("="*60)

# Tim model co test_f1 cao nhat
best_row = results_df.loc[results_df["test_f1"].idxmax()]
best_name = best_row["model_name"]
best_f1   = best_row["test_f1"]

print(f"\nModel tot nhat: {best_name}")
print(f"  Test F1 cao nhat: {best_f1:.4f}")

# Lay F1 cua baseline
baseline_row = results_df[results_df["mode"] == "baseline"]
if not baseline_row.empty:
    baseline_f1 = baseline_row.iloc[0]["test_f1"]
    print(f"\nBaseline Test F1: {baseline_f1:.4f}")

    # Ket qua CA (last)
    ca_last = results_df[results_df["mode"] == "ca_last"]
    if not ca_last.empty:
        ca_last_f1 = ca_last.iloc[0]["test_f1"]
        if ca_last_f1 > baseline_f1:
            print(f"  Coordinate Attention (last): {ca_last_f1:.4f} > baseline -> "
                  "CA giup giu thong tin vi tri theo chieu H va W, "
                  "cai thien kha nang phan biet vung benh tren X-ray.")
        else:
            print(f"  Coordinate Attention (last): {ca_last_f1:.4f} <= baseline -> "
                  "CA khong cai thien ro o vi tri nay.")

    # Ket qua CBAM (last)
    cbam_last = results_df[results_df["mode"] == "cbam_last"]
    if not cbam_last.empty:
        cbam_last_f1 = cbam_last.iloc[0]["test_f1"]
        if cbam_last_f1 > baseline_f1:
            print(f"  CBAM (last): {cbam_last_f1:.4f} > baseline -> "
                  "CBAM giup chon loc kenh va khong gian, "
                  "nhan manh vung quan trong tren anh X-ray.")
        else:
            print(f"  CBAM (last): {cbam_last_f1:.4f} <= baseline -> "
                  "CBAM khong cai thien o vi tri nay.")

    # Ket qua CA+CBAM (last)
    ca_cbam_last = results_df[results_df["mode"] == "ca_cbam_last"]
    if not ca_cbam_last.empty:
        ca_cbam_f1 = ca_cbam_last.iloc[0]["test_f1"]
        if ca_cbam_f1 > max(baseline_f1,
                             ca_last.iloc[0]["test_f1"] if not ca_last.empty else 0,
                             cbam_last.iloc[0]["test_f1"] if not cbam_last.empty else 0):
            print(f"  CA + CBAM (last): {ca_cbam_f1:.4f} -> "
                  "Ket hop CA va CBAM mang lai hieu qua tot nhat.")
        elif ca_cbam_f1 <= baseline_f1:
            print(f"  CA + CBAM (last): {ca_cbam_f1:.4f} <= baseline -> "
                  "Co the do EfficientNet-B1 da co SE-block ben trong, "
                  "them CA+CBAM du thua va gay nhieu.")

    # Ca stage so voi last
    stage_f1s = results_df[results_df["attention_position"] == "stage"]["test_f1"]
    if not stage_f1s.empty:
        best_stage_f1 = stage_f1s.max()
        if best_stage_f1 > results_df[results_df["attention_position"] == "last"]["test_f1"].max():
            print(f"\n  Attention chen vao giua stage ({best_stage_f1:.4f}) tot hon dua vao cuoi backbone "
                  "-> Chen attention vao stage trung gian giup mo hinh hoc dac trung phuc tap hon.")
        else:
            print(f"\n  Attention cuoi backbone tot hon hoac tuong duong chen vao stage -> "
                  "Voi EfficientNet-B1, them attention cuoi backbone la du.")

    if best_name == "EfficientNet_B1_Baseline":
        print("\nNhan xet: EfficientNet-B1 goc da dat ket qua tot nhat. "
              "Dieu nay co the do dataset khong du lon de attention "
              "phat huy hieu qua, hoac EfficientNet-B1 da co cac "
              "co che SE va compound scaling du manh.")

# In bang ket qua day du
print("\nBang ket qua day du:")
print(results_df[[
    "model_name", "test_acc", "test_precision",
    "test_recall", "test_f1", "inference_time_ms"
]].to_string(index=False))